# C12_E01 — Cascada en intercambiador térmico

**Capítulo:** 12 — Estrategias avanzadas  
**Identificador:** `C12_E01`  
**Objetivo pedagógico:** Diseñar y validar una estructura cascada con sintonía interno→externo.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Intercambiador térmico calentado por vapor: cascada caudal-vapor / temperatura-salida.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Lazo interno (caudal) y externo (temperatura)

In [1]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import os

# Lazo interno: caudal con dinámica rápida (tau=1)
Gi = ct.tf([1.0], [1.0, 1.0])
Ci = ct.tf([2.0, 0.5], [1.0, 0.0])      # PI rápido
Ti_int = ct.feedback(Ci*Gi, 1)          # FDT lazo interno cerrado

# Lazo externo: temperatura con dinámica más lenta (tau=10)
Ge = ct.tf([1.0], [10.0, 1.0])
Ce = ct.tf([0.4, 0.04], [1.0, 0.0])

L_externo = Ce*Ti_int*Ge
T_externo_cerrado = ct.feedback(L_externo, 1)
print("Polos del lazo externo cerrado (cascada):", ct.poles(T_externo_cerrado))

Polos del lazo externo cerrado (cascada): [-2.79505039+0.j -0.16031579+0.j -0.1       +0.j -0.04463382+0.j]


## 2. Comparativa cascada vs lazo único externo

In [2]:
# Lazo único: solo PI externo sobre la planta total Gi*Ge (sin lazo interno)
G_total = Gi*Ge
T_unico = ct.feedback(Ce*G_total, 1)

t = np.linspace(0, 60, 600)
_, y_cas = ct.step_response(T_externo_cerrado, t)
_, y_uni = ct.step_response(T_unico, t)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_cas, label="Cascada (interno + externo)")
ax.plot(t, y_uni, '--', label="Lazo único externo")
ax.axhline(1.0, ls=':', color='gray')
ax.set_xlabel("t (s)"); ax.set_ylabel("y/SP")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C12_E01 — Cascada en intercambiador (respuesta a escalón en consigna)")
out = '../../figures/cap12/fig_C12_F01_cascada.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

La cascada produce una respuesta más rápida y menos sobredisparo en seguimiento. Su ventaja real, no mostrada aquí, es el rechazo a perturbaciones que entran en el lazo interno (variaciones de presión de vapor, en el ejemplo industrial): el lazo interno las absorbe antes de que afecten a la temperatura. Regla de sintonía: lazo interno al menos 5 veces más rápido que el externo.